# Walk-through: detection limits of cfDNA methylation scores

This notebook reads the summary tables in `results/` and needs no downloaded data. It runs in
seconds. The heavy steps (simulation, model training) are the scripts in `scripts/`.

Terms used below:

- **f**: the tumour-cell (or colon) DNA fraction of a simulated sample.
- **LoD, limit of detection**: the lowest f detected in at least 95 % of tests.
- **FPR, false-positive rate**: the share of held-out healthy blanks scored above the threshold. The rule needs 5 % or less.
- **HI1**: one healthy individual whose blank is an outlier; results are shown without it.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
res = root / "results"
pd.set_option("display.precision", 3)

## Phase 1: reproduce Moss et al. 2018, Fig. 7d

Correlation of the observed colon fraction with the nominal plasma share of each dilution mix.

In [2]:
pd.read_csv(res / "phase1/fig7d_stats.tsv", sep="\t")

,set,x,n,r,p,slope,intercept
0,all 8,p_nominal,8,0.940,5.228e-04,1.247,0.013
1,all 8,expected,8,0.938,5.764e-04,1.874,-0.013
2,without mix_7,p_nominal,7,0.958,6.698e-04,1.232,0.017
3,without mix_7,expected,7,0.957,7.239e-04,1.852,-0.009
4,5 mixes only,p_nominal,5,0.854,6.538e-02,1.318,0.008
5,5 mixes only,expected,5,0.849,6.904e-02,1.973,-0.019
6,"5 mixes only, without mix_7",p_nominal,4,0.799,2.013e-01,0.961,0.035
7,"5 mixes only, without mix_7",expected,4,0.794,2.064e-01,1.431,0.016


## Phase 2: limit of blank and limit of detection on the arrays

Leave-one-blank-out: each of the 12 blanks in turn is the background, and its limit of blank
(LoB) comes from the other blanks. `f_needed_opt1` and `f_needed_opt2` are the colon
fractions at which that background crosses the LoB (option 1: colon-cell spike; option 2:
pl806 plasma spike). Their 95th percentile is about 2.1 % here (linear interpolation).
The script's EP17 rank rule gives 2.14–2.17 % (`results/phase2/run.log`). Scoring every spike
replicate and noise state gives the README range, 2.0–2.3 % colon DNA. HI1 is the one blank
that is a false positive.

In [3]:
loo = pd.read_csv(res / "phase2/leave_one_blank_out.tsv", sep="	")
display_cols = ["label", "colon", "lob_without", "f_needed_opt1", "f_needed_opt2", "false_positive"]
print(loo[["f_needed_opt1", "f_needed_opt2"]].quantile([0.5, 0.95]).mul(100).round(2))
loo[display_cols]

      f_needed_opt1  f_needed_opt2
0.50           1.56           1.54
0.95           2.10           2.07


,label,colon,lob_without,f_needed_opt1,f_needed_opt2,false_positive
0,Young M1,0.007,0.027,0.020,0.020,False
1,Young M2,0.016,0.027,0.011,0.011,False
2,Young F1,0.005,0.027,0.022,0.022,False
3,Young F2,0.010,0.027,0.017,0.017,False
4,Old M1,0.013,0.027,0.014,0.014,False
5,Old M2,0.010,0.027,0.017,0.017,False
6,Old F1,0.010,0.027,0.017,0.017,False
7,Old F2,0.007,0.027,0.020,0.020,False
8,HI1,0.027,0.016,0.000,0.000,True
9,HI2,0.014,0.026,0.012,0.012,False


The split tables in `results/phase2/lod.tsv` are kept for reference. The default split
evaluates only three similar backgrounds (HI2–HI4); the reverse split gives 22 % false
positives at f = 0, so its LoD is not valid.

## Phase 3: sequencing depth

LoD (interpolated) of the valid-rule rows with HI1 excluded, plasma spike, no input cap.
Rows whose held-out FPR is above 5 % are masked.

In [4]:
p3 = pd.read_csv(res / "phase3/lod.tsv", sep="\t")
sel = p3[(p3.rule == "valid") & (p3.hi1 == "without") & (p3.kind == "plasma")
         & (p3.cap == "none") & (p3.bg == "person")]
lod3 = sel.assign(lod=sel.lod_interp.where(sel.specificity_ok) * 100)
lod3.pivot_table(index=["score", "markers"], columns="depth", values="lod")

depth          10     30     100    300    1000   3000   10000
score markers                                                 
LLR   100      2.740  1.951  1.683  1.590  1.579  1.595  1.645
      500      2.659  2.463  2.411  2.400  2.400  2.400  2.400
      all      2.854  2.759  2.707  2.698  2.683  2.683  2.683
NNLS  500        NaN    NaN  2.791  2.667  2.570  2.532  2.469
      all      2.758  1.852  1.413  1.306  1.250  1.205  1.200

![LoD against sequencing depth](../figures/phase3_depth_lod.png)

## Phase 3b: logistic regression and XGBoost against untrained scores

LoD in % tumour-cell DNA, HI1 excluded, primary rows only (no `_refit` diagnostics). A cell
is masked when the largest held-out FPR is above 5 %. The array-track FPR rests on 11 people:
0 of 11 false positives has a one-sided 95 % upper bound of 24 %.

In [5]:
p3b = pd.read_csv(res / "phase3b/lod.tsv", sep="\t")
prim = p3b[(p3b.hi1 == "without") & ~p3b.model.str.contains("_refit")]
prim = prim.assign(lod=(prim.lod_interp * 100).where(prim.specificity_ok))
prim.pivot_table(index=["features", "model"], columns="track", values="lod")[
    ["array", "30", "100", "300"]]

track           array     30    100    300
features model                            
atlas    LLR      NaN  3.313  3.261  3.552
         LR_L1  1.624  4.416  3.392    NaN
         LR_L2  2.715  2.925  2.512  2.619
         NNLS   1.906  2.740  1.972  1.944
         XGB    2.667  2.952  2.414  2.244
tumour   LLR      NaN  4.882  4.763  4.589
         LR_L1  2.668  4.445    NaN  2.847
         LR_L2  2.966  4.249    NaN  2.955
         XGB      NaN  4.093  2.694  2.064

The LR_L1 array value above is for the main seed only. An independent re-run over 7 seeds
(not scripted here) gave 0.9 to 1.7 % (median 1.5 %), so it is labelled unverified. NNLS gives 1.9–2.0 %.

![Sensitivity against tumour fraction](../figures/phase3b_classifiers.png)

## Limitations

- Every FPR rests on 11 people in 7 groups.
- The LR_L1 LoD depends on the random seed.
- Reads are simulated as Binomial(D, beta). Real reads are the next phase.